# FlyGraph Attention vs SmolLM2-135M — corrected experiment

This notebook replaces every SmolLM2-135M self-attention layer with **FlyGraph causal linear attention** while keeping the pretrained embedding, RMSNorm, MLP/residual stack, and pretrained Q/K/V/O initialization.

The training cell streams live subprocess output and adds a heartbeat with the current stage, progress, elapsed time, and an ETA once enough training updates have been observed.


In [1]:
#@title 1. Clone/update repository and install
import pathlib, subprocess, sys

REPO_DIR = pathlib.Path("/content/TinyCeNN-LM")
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/main"], check=True)
else:
    subprocess.run(
        ["git", "clone", "https://github.com/vtavakkoli/TinyCeNN-LM.git", str(REPO_DIR)],
        check=True,
    )

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR),
    "transformers>=4.56", "datasets>=3.0", "accelerate>=1.0",
    "huggingface_hub>=0.34", "pandas>=2.0", "requests>=2.31", "tqdm>=4.66"
], check=True)

print("Repository ready:", REPO_DIR)


Repository ready: /content/TinyCeNN-LM


In [2]:
#@title 2. Choose experiment size
RUN_MODE = "quick"  #@param ["quick", "strong"]
SEQ_LEN = 128       #@param {type:"integer"}
FEATURE_DIM = 256   #@param {type:"integer"}
MAX_EDGES = 2048    #@param {type:"integer"}
GRAPH_STEPS = 1     #@param {type:"integer"}
RUN_REWIRED_CONTROL = True  #@param {type:"boolean"}

OUTPUT_DIR = REPO_DIR / "results" / "flygraph_attention_smollm2_135m"
print("Output:", OUTPUT_DIR)


Output: /content/TinyCeNN-LM/results/flygraph_attention_smollm2_135m


In [3]:
#@title 3. Train/evaluate — LIVE status, progress and ETA
import os, re, time, queue, threading, subprocess

cmd = [
    sys.executable, "-u",
    str(REPO_DIR / "scripts" / "run_smollm2_fly_graph_attention.py"),
    "--run-mode", RUN_MODE,
    "--seq-len", str(SEQ_LEN),
    "--feature-dim", str(FEATURE_DIM),
    "--max-edges", str(MAX_EDGES),
    "--graph-steps", str(GRAPH_STEPS),
    "--output-dir", str(OUTPUT_DIR),
]
if RUN_REWIRED_CONTROL:
    cmd.append("--rewired")

TRAIN_UPDATES = 300 if RUN_MODE == "quick" else 2500
MODEL_COUNT = 2 if RUN_REWIRED_CONTROL else 1
STATUS_EVERY = 10.0

def fmt_time(seconds):
    if seconds is None or seconds < 0 or not float(seconds) < float("inf"):
        return "calculating..."
    seconds = int(seconds)
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f"{h:d}h {m:02d}m {s:02d}s" if h else f"{m:d}m {s:02d}s"

def make_bar(frac, width=28):
    frac = max(0.0, min(1.0, float(frac)))
    n = int(round(frac * width))
    return "#" * n + "-" * (width - n)

print("=" * 88)
print("FlyGraph / SmolLM2-135M experiment")
print(f"Mode: {RUN_MODE} | updates/model: {TRAIN_UPDATES} | models: {MODEL_COUNT}")
print(f"Sequence: {SEQ_LEN} | features: {FEATURE_DIM} | max edges: {MAX_EDGES} | graph steps: {GRAPH_STEPS}")
print("ETA is calibrated from the actual GPU speed after training starts.")
print("Command:", " ".join(cmd))
print("=" * 88, flush=True)

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["TQDM_MININTERVAL"] = "1"

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

lines = queue.Queue()
DONE = object()

def reader():
    try:
        for line in iter(proc.stdout.readline, ""):
            lines.put(line.rstrip("\n"))
    finally:
        lines.put(DONE)

threading.Thread(target=reader, daemon=True).start()

progress_re = re.compile(r"^\s*(biological|rewired)\s+(\d+)\s*/\s*(\d+)\s+loss=")
start = time.time()
last_heartbeat = 0.0
stage = "setup: FlyWire graph -> SmolLM2 load -> FineWeb-Edu preparation"
current_model = None
model_start = {}
last_update = {"biological": 0, "rewired": 0}
seconds_per_update = {}
reader_finished = False

while True:
    now = time.time()
    try:
        item = lines.get(timeout=1.0)
    except queue.Empty:
        item = None

    if item is DONE:
        reader_finished = True
    elif item:
        print(item, flush=True)
        low = item.lower()

        if "flywire graph" in low:
            stage = "downloading/caching FlyWire connectome"
        elif "degree pass" in low:
            stage = "scanning FlyWire node strengths"
        elif "edge pass" in low:
            stage = "extracting biological FlyWire subgraph"
        elif "pure replacement check" in low:
            stage = "validating pure attention replacement"
        elif low.strip() == "summary":
            stage = "final report"
        elif "saved:" in low:
            stage = "saving complete"

        match = progress_re.search(item)
        if match:
            model = match.group(1)
            update = int(match.group(2))
            total = int(match.group(3))
            current_model = model
            stage = f"training {model} FlyGraph attention"
            if model not in model_start:
                model_start[model] = now
            last_update[model] = update
            elapsed_model = max(now - model_start[model], 1e-6)
            seconds_per_update[model] = elapsed_model / max(update, 1)

    if now - last_heartbeat >= STATUS_EVERY:
        elapsed = now - start
        overall_total = TRAIN_UPDATES * MODEL_COUNT
        bio_done = last_update["biological"]
        rew_done = last_update["rewired"]
        overall_done = bio_done + (rew_done if RUN_REWIRED_CONTROL else 0)

        eta = None
        if current_model and current_model in seconds_per_update:
            sec_per = seconds_per_update[current_model]
            if current_model == "biological":
                remaining_updates = TRAIN_UPDATES - bio_done
                if RUN_REWIRED_CONTROL:
                    remaining_updates += TRAIN_UPDATES
            else:
                remaining_updates = TRAIN_UPDATES - rew_done
            eta = max(0.0, remaining_updates * sec_per)

        frac = overall_done / max(overall_total, 1)
        print()
        print(f"STATUS | {stage}")
        print(f"[{make_bar(frac)}] {100*frac:5.1f}% training updates ({overall_done}/{overall_total})")
        print(f"elapsed: {fmt_time(elapsed)} | ETA: {fmt_time(eta)}")
        if current_model and current_model in seconds_per_update:
            rate = 1.0 / max(seconds_per_update[current_model], 1e-9)
            print(f"current speed: {rate:.3f} updates/s | model: {current_model}")
        else:
            print("waiting for first training update to calibrate ETA...")
        print(flush=True)
        last_heartbeat = now

    if proc.poll() is not None and reader_finished and lines.empty():
        break

return_code = proc.wait()
elapsed_total = time.time() - start
print("=" * 88)
print(f"Process finished with exit code {return_code} in {fmt_time(elapsed_total)}")
print("=" * 88)

if return_code != 0:
    raise subprocess.CalledProcessError(return_code, cmd)


FlyGraph / SmolLM2-135M experiment
Mode: quick | updates/model: 300 | models: 2
Sequence: 128 | features: 256 | max edges: 2048 | graph steps: 1
ETA is calibrated from the actual GPU speed after training starts.
Command: /usr/bin/python3 -u /content/TinyCeNN-LM/scripts/run_smollm2_fly_graph_attention.py --run-mode quick --seq-len 128 --feature-dim 256 --max-edges 2048 --graph-steps 1 --output-dir /content/TinyCeNN-LM/results/flygraph_attention_smollm2_135m --rewired

STATUS | setup: FlyWire graph -> SmolLM2 load -> FineWeb-Edu preparation
[----------------------------]   0.0% training updates (0/600)
elapsed: 0m 00s | ETA: calculating...
waiting for first training update to calibrate ETA...


STATUS | setup: FlyWire graph -> SmolLM2 load -> FineWeb-Edu preparation
[----------------------------]   0.0% training updates (0/600)
elapsed: 0m 10s | ETA: calculating...
waiting for first training update to calibrate ETA...


STATUS | setup: FlyWire graph -> SmolLM2 load -> FineWeb-Edu prepara

In [4]:
#@title 4. Show results
import json
import pandas as pd
from IPython.display import display

summary = pd.read_csv(OUTPUT_DIR / "summary.csv", index_col=0)
report = json.loads((OUTPUT_DIR / "report.json").read_text())

display(summary)

print("\nATTENTION CHECK")
print("pure_attention_replacement:", report["pure_attention_replacement"])
print("standard_llama_attention_modules:", report["standard_llama_attention_modules"])
print("streaming_full_max_abs_logit_diff:", report["streaming_full_max_abs_logit_diff"])

print("\nKEY METRICS")
for key in (
    "fly_ce_gap_vs_smollm2",
    "fly_ppl_ratio_vs_smollm2",
    "decode_speed_ratio_fly_over_smollm2",
    "biological_topology_ce_gain",
    "biological_topology_ppl_gain_pct",
):
    if key in report:
        print(f"{key}: {report[key]}")

print("\nHow to read this:")
print("1) CE/PPL should be much better than the old FlyCeNN result.")
print("2) pure_attention_replacement must be True and standard_llama_attention_modules must be 0.")
print("3) streaming_full_max_abs_logit_diff should be tiny.")
print("4) biological_topology_ce_gain > 0 means real FlyWire wiring beat the rewired control.")
print("5) If quick mode closes much of the SmolLM2 gap, rerun with RUN_MODE='strong'.")


,ce,perplexity,weight_mb,prefill_tokens_s,prefill_peak_extra_mb,decode_tokens_s,decode_peak_extra_mb,teacher_kl
SmolLM2-135M,3.053056,21.179977,256.567017,2759.681918,14.953125,24.790114,2.104004,NaN
FlyGraph biological,6.443463,628.579562,257.739235,838.348178,833.005371,12.879069,0.977051,4.557221
FlyGraph rewired,6.467811,644.072557,NaN,NaN,NaN,NaN,NaN,4.617412



ATTENTION CHECK
pure_attention_replacement: True
standard_llama_attention_modules: 0
streaming_full_max_abs_logit_diff: 0.25

KEY METRICS
fly_ce_gap_vs_smollm2: 3.390406350294749
fly_ppl_ratio_vs_smollm2: 29.678009488379058
decode_speed_ratio_fly_over_smollm2: 0.5195243921222812
biological_topology_ce_gain: 0.024348775545756318
biological_topology_ppl_gain_pct: 2.4054735450392855

How to read this:
1) CE/PPL should be much better than the old FlyCeNN result.
2) pure_attention_replacement must be True and standard_llama_attention_modules must be 0.
3) streaming_full_max_abs_logit_diff should be tiny.
4) biological_topology_ce_gain > 0 means real FlyWire wiring beat the rewired control.
5) If quick mode closes much of the SmolLM2 gap, rerun with RUN_MODE='strong'.


In [7]:
# Fix TinyCeNN-LM import path
import sys
from pathlib import Path

REPO_DIR = Path("/content/TinyCeNN-LM")
SRC_DIR = REPO_DIR / "src"

assert REPO_DIR.exists(), f"Repository not found: {REPO_DIR}"
assert SRC_DIR.exists(), f"src directory not found: {SRC_DIR}"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Repository:", REPO_DIR)
print("Python package path:", SRC_DIR)

# Now this works
from tinycenn_lm.smollm2_fly_graph_attention import (
    FlyGraphAttentionConfig,
    replace_all_attention_with_fly,
    assert_pure_fly_attention,
    set_fly_streaming,
)

print("✓ tinycenn_lm imported successfully")

Repository: /content/TinyCeNN-LM
Python package path: /content/TinyCeNN-LM/src
✓ tinycenn_lm imported successfully


In [8]:
#@title 5. Load trained FlyGraph model and test prompts

import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from tinycenn_lm.smollm2_fly_graph_attention import (
    FlyGraphAttentionConfig,
    replace_all_attention_with_fly,
    assert_pure_fly_attention,
    set_fly_streaming,
)

BASE_MODEL = "HuggingFaceTB/SmolLM2-135M"
STATE_FILE = OUTPUT_DIR / "biological_fly_attention.pt"
REPORT_FILE = OUTPUT_DIR / "report.json"

assert STATE_FILE.exists(), f"Missing: {STATE_FILE}"
assert REPORT_FILE.exists(), f"Missing: {REPORT_FILE}"

report = json.loads(REPORT_FILE.read_text())
cfg_saved = report["config"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = (
    torch.bfloat16
    if device.type == "cuda" and torch.cuda.is_bf16_supported()
    else torch.float16
    if device.type == "cuda"
    else torch.float32
)

print("Loading trained FlyGraph checkpoint...")
state = torch.load(STATE_FILE, map_location="cpu", weights_only=True)

# Recover the exact biological graph directly from the saved checkpoint.
prefix = "model.layers.0.self_attn.graph."
src = state[prefix + "src"]
dst = state[prefix + "dst"]
base_weight = state[prefix + "base_weight"]

print(f"Graph: {len(torch.unique(torch.cat([src, dst])))} active nodes")
print(f"Edges: {len(src):,}")
print(f"Feature dim: {cfg_saved['feature_dim']}")
print(f"Graph steps: {cfg_saved['graph_steps']}")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)

fly_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=dtype if device.type == "cuda" else torch.float32,
).to(device)

fly_cfg = FlyGraphAttentionConfig(
    feature_dim=int(cfg_saved["feature_dim"]),
    graph_steps=int(cfg_saved["graph_steps"]),
    graph_gate_init=0.10,
    content_gate_init=0.25,
    feature_seed=7331,
)

replace_all_attention_with_fly(
    fly_model,
    fly_cfg,
    src,
    dst,
    base_weight,
)

load_result = fly_model.load_state_dict(state, strict=False)

missing_attention = [
    k for k in load_result.missing_keys
    if ".self_attn." in k
]

if missing_attention:
    raise RuntimeError(
        "Missing trained FlyGraph weights:\n" +
        "\n".join(missing_attention[:20])
    )

assert_pure_fly_attention(fly_model)

fly_model.eval()

print("✓ Model loaded")
print("✓ Standard Llama attention remaining: 0")


# ------------------------------------------------------------
# Streaming generation
# ------------------------------------------------------------

@torch.no_grad()
def generate_flygraph(
    prompt,
    max_new_tokens=80,
    temperature=0.75,
    top_p=0.90,
):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=True,
    )

    input_ids = inputs["input_ids"].to(device)

    # Reset recurrent/linear-attention state.
    set_fly_streaming(
        fly_model,
        True,
        reset=True,
    )

    # Prefill complete prompt once.
    positions = torch.arange(
        input_ids.size(1),
        device=device,
    ).unsqueeze(0)

    with torch.autocast(
        device_type=device.type,
        dtype=dtype,
        enabled=device.type == "cuda",
    ):
        out = fly_model(
            input_ids=input_ids,
            position_ids=positions,
            use_cache=False,
            return_dict=True,
        )

    generated = input_ids.clone()
    logits = out.logits[:, -1, :]

    current_position = input_ids.size(1)

    for _ in range(max_new_tokens):

        # Temperature
        logits = logits.float() / max(temperature, 1e-5)

        probs = torch.softmax(logits, dim=-1)

        # Top-p nucleus sampling
        sorted_probs, sorted_idx = torch.sort(
            probs,
            descending=True,
        )

        cumulative = torch.cumsum(sorted_probs, dim=-1)

        remove = cumulative > top_p

        # Always retain highest-probability token
        remove[:, 0] = False

        sorted_probs = sorted_probs.masked_fill(
            remove,
            0.0,
        )

        sorted_probs = (
            sorted_probs /
            sorted_probs.sum(dim=-1, keepdim=True)
        )

        sampled_index = torch.multinomial(
            sorted_probs,
            num_samples=1,
        )

        next_token = sorted_idx.gather(
            -1,
            sampled_index,
        )

        generated = torch.cat(
            [generated, next_token],
            dim=1,
        )

        if (
            tokenizer.eos_token_id is not None
            and next_token.item() == tokenizer.eos_token_id
        ):
            break

        # Only feed the NEW token.
        position_ids = torch.tensor(
            [[current_position]],
            device=device,
        )

        with torch.autocast(
            device_type=device.type,
            dtype=dtype,
            enabled=device.type == "cuda",
        ):
            out = fly_model(
                input_ids=next_token,
                position_ids=position_ids,
                use_cache=False,
                return_dict=True,
            )

        logits = out.logits[:, -1, :]
        current_position += 1

    set_fly_streaming(
        fly_model,
        False,
        reset=True,
    )

    return tokenizer.decode(
        generated[0],
        skip_special_tokens=True,
    )


# ------------------------------------------------------------
# Test prompts
# ------------------------------------------------------------

PROMPTS = [
    "Artificial intelligence can",
    "The city of Vienna is",
    "A neural network learns",
    "The future of efficient computing",
    "Machine learning models are useful because",
    "The human brain processes information by",
]

for i, prompt in enumerate(PROMPTS, 1):

    print("\n" + "=" * 100)
    print(f"PROMPT {i}: {prompt}")
    print("=" * 100)

    result = generate_flygraph(
        prompt,
        max_new_tokens=80,
        temperature=0.7,
        top_p=0.9,
    )

    print(result)

Loading trained FlyGraph checkpoint...
Graph: 256 active nodes
Edges: 2,048
Feature dim: 256
Graph steps: 1


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

✓ Model loaded
✓ Standard Llama attention remaining: 0

PROMPT 1: Artificial intelligence can
Artificial intelligence can.





The New York:

  • "The workbook



For a.

The Internet in which is a.
The New definition
N.



the digital news
The Office
will do you can.

 ─.
 deficienton 102. per

                                                                                 19902.

PROMPT 2: The city of Vienna is
The city of Vienna is the city of the Vis-in the majority of the world's. In the 'The city-city of the best city of






















But in the first












 ●. The third.
 popular name of the city-


PROMPT 3: A neural network learns
A neural network learns about 10, and 20, 1, 2


















â€










 Patel, 160
 N/t






 car - 2.
 fancy
 enthus (15
 PA: 

PROMPT 4: The future of efficient computing
The future of efficient computing and machine will be the heart-from the brain.




















 H. The official
The National Health and the head to


